# AlpCAN BT-3: Nodul Malignite Siniflandirmasi

**Amac:** LIDC-IDRI uzerinde coklu ozellik (boyut, morfoloji, yogunluk, doku) tabanli malignite siniflandirmasi.
NB07 (Karakterizasyon) boyut bazli proxy etiketi kullaniyordu.
Bu notebook morfolojik ve radyomik ozellikler ile daha guclu bir malignite tahmini yapar.

**Hedef:** Binary AUC > 0.85, 5-sinif accuracy > 0.55

**Model:** EfficientNet-B0 + Tabular Feature Fusion
**Dataset:** zhangweiled/lidcidri (LIDC-IDRI slices)
**GPU:** Kaggle T4 16GB
**Egitim:** 50 epoch, fold 0

**Iyilestirmeler (NB07'ye gore):**
1. Boyut + morfoloji + yogunluk + doku (GLCM) ozellikleri ile multi-feature malignite proxy
2. EfficientNet-B0 (hafif) + tabular feature fusion mimarisi
3. Binary + 5-sinif ordinal cikis (Lung-RADS uyumlu)
4. Focal Loss + Label Smoothing CE
5. Ozellik onemi analizi

In [ ]:
!pip install -q scikit-image

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, accuracy_score
)
from sklearn.preprocessing import StandardScaler

from skimage.measure import regionprops, label as sk_label
from skimage.feature import graycomatrix, graycoprops
from scipy import ndimage

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# --- Konfigürasyon ---
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LIDC-IDRI dataset -- rglob fallback
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")
if not DATA_DIR.exists():
    for candidate in INPUT_DIR.rglob("LIDC-IDRI-0001"):
        DATA_DIR = candidate.parent
        break
    if not DATA_DIR.exists():
        raise FileNotFoundError("LIDC-IDRI dataset bulunamadi!")

print(f"Dataset: {DATA_DIR}")

# Hasta dizinleri -- LIDC-IDRI- prefiksi ile filtrele
patient_dirs = sorted([
    d for d in DATA_DIR.iterdir()
    if d.is_dir() and d.name.startswith("LIDC-IDRI-")
])
print(f"Toplam hasta: {len(patient_dirs)}")

# Sabitler
PATCH_SIZE = 128
BATCH_SIZE = 32
NUM_EPOCHS = 100      # 50 → 100 (AUC hala artiyordu)
SEED = 42
N_MALIGNANCY_CLASSES = 5

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Patch boyutu: {PATCH_SIZE}, Batch: {BATCH_SIZE}, Epoch: {NUM_EPOCHS}")

---
## 2. Ozellik Cikarimi (Feature Extraction)

Her nodul icin 5 kategori ozellik cikarilir:
1. **Boyut:** cap, alan
2. **Morfoloji:** compactness, eccentricity, solidity, margin sharpness
3. **Yogunluk:** nodul mean/std, cevre mean/std, kontrast orani
4. **Doku (GLCM):** homogeneity, contrast, energy, correlation
5. **Annotator:** annotator sayisi, dilim sayisi

Bu ozellikler birlestirilip agirlikli skor ile malignite proxy etiketi olusturulur.

In [ ]:
def compute_morphology_features(binary_mask):
    """Maske morfoloji ozellikleri.
    
    Returns:
        dict: compactness, eccentricity, solidity, perimeter
    """
    labeled = sk_label(binary_mask.astype(np.uint8))
    props_list = regionprops(labeled)
    if not props_list:
        return {
            'compactness': 1.0, 'eccentricity': 0.0,
            'solidity': 1.0, 'perimeter': 0.0
        }
    # En buyuk bolgeyi sec
    props = max(props_list, key=lambda p: p.area)
    area = props.area
    perimeter = props.perimeter if props.perimeter > 0 else 1e-6
    compactness = (4 * np.pi * area) / (perimeter ** 2)
    compactness = min(compactness, 1.0)  # Numerik kararlillik
    eccentricity = props.eccentricity
    solidity = props.solidity
    return {
        'compactness': float(compactness),
        'eccentricity': float(eccentricity),
        'solidity': float(solidity),
        'perimeter': float(perimeter),
    }


def compute_intensity_features(image, binary_mask):
    """Nodul ve cevre yogunluk ozellikleri.
    
    Cevre bolgesi: maskeyi 5 piksel genisleterek elde edilir.
    Kontrast orani: nodul mean / cevre mean.
    """
    nodule_pixels = image[binary_mask > 0]
    if len(nodule_pixels) == 0:
        return {
            'intensity_mean': 0.0, 'intensity_std': 0.0,
            'surround_mean': 0.0, 'surround_std': 0.0,
            'contrast_ratio': 0.0,
        }
    # Cevre bolgesi: dilate - original mask
    dilated = ndimage.binary_dilation(binary_mask, iterations=5)
    surround_mask = dilated & (~binary_mask)
    surround_pixels = image[surround_mask > 0]
    
    nod_mean = float(np.mean(nodule_pixels))
    nod_std = float(np.std(nodule_pixels))
    sur_mean = float(np.mean(surround_pixels)) if len(surround_pixels) > 0 else nod_mean
    sur_std = float(np.std(surround_pixels)) if len(surround_pixels) > 0 else nod_std
    contrast = nod_mean / (sur_mean + 1e-6)
    
    return {
        'intensity_mean': nod_mean,
        'intensity_std': nod_std,
        'surround_mean': sur_mean,
        'surround_std': sur_std,
        'contrast_ratio': float(contrast),
    }


def compute_margin_sharpness(image, binary_mask):
    """Maske kenarindaki gradient buyuklugu (keskinlik).
    
    Yuksek gradient = keskin kenar (benign egilim)
    Dusuk gradient = belirsiz kenar (malign egilim)
    """
    # Sobel gradientleri
    gx = ndimage.sobel(image.astype(np.float64), axis=1)
    gy = ndimage.sobel(image.astype(np.float64), axis=0)
    gradient_mag = np.sqrt(gx**2 + gy**2)
    
    # Kenar piksellerini bul (erozyon ile)
    eroded = ndimage.binary_erosion(binary_mask, iterations=1)
    border = binary_mask & (~eroded)
    border_pixels = gradient_mag[border > 0]
    
    if len(border_pixels) == 0:
        return 0.0
    return float(np.mean(border_pixels))


def compute_glcm_features(image, binary_mask):
    """GLCM doku ozellikleri (Gray-Level Co-occurrence Matrix).
    
    4 yonde ortalama alinir: 0, 45, 90, 135 derece.
    """
    # Nodul bolgesini crop et
    rows = np.where(np.any(binary_mask, axis=1))[0]
    cols = np.where(np.any(binary_mask, axis=0))[0]
    if len(rows) == 0 or len(cols) == 0:
        return {
            'glcm_homogeneity': 0.0, 'glcm_contrast': 0.0,
            'glcm_energy': 0.0, 'glcm_correlation': 0.0,
        }
    
    crop = image[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    crop_mask = binary_mask[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    
    # Maske disini sifirla
    crop = crop.copy()
    crop[crop_mask == 0] = 0
    
    # Minimum boyut kontrolu
    if crop.shape[0] < 2 or crop.shape[1] < 2:
        return {
            'glcm_homogeneity': 0.0, 'glcm_contrast': 0.0,
            'glcm_energy': 0.0, 'glcm_correlation': 0.0,
        }
    
    # GLCM hesapla (4 yon, mesafe=1)
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(
        crop, distances=[1], angles=angles,
        levels=256, symmetric=True, normed=True
    )
    
    # Ozellikler -- 4 yonun ortalamasini al
    homogeneity = float(np.mean(graycoprops(glcm, 'homogeneity')))
    contrast = float(np.mean(graycoprops(glcm, 'contrast')))
    energy = float(np.mean(graycoprops(glcm, 'energy')))
    correlation = float(np.mean(graycoprops(glcm, 'correlation')))
    
    return {
        'glcm_homogeneity': homogeneity,
        'glcm_contrast': contrast,
        'glcm_energy': energy,
        'glcm_correlation': correlation,
    }


# ---------------------------------------------------------------
# Ana ozellik cikarimi dongusu
# ---------------------------------------------------------------
print("Ozellik cikarimi basliyor...")
t0 = time.time()
nodule_data = []

for pi, pdir in enumerate(patient_dirs):
    nodule_dirs = sorted([d for d in pdir.iterdir() if d.is_dir()])
    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs = sorted([
            d for d in ndir.iterdir()
            if d.is_dir() and d.name.startswith("mask")
        ])
        if not img_dir.exists() or not mask_dirs:
            continue

        img_files = sorted(img_dir.glob("*.png"))
        if not img_files:
            continue

        # Orta dilim
        mid_idx = len(img_files) // 2
        mid_img_path = img_files[mid_idx]

        # Goruntu oku
        img = np.array(Image.open(mid_img_path).convert('L'))

        # Tum annotator maskeleri -> konsensus
        mask_areas = []
        combined_mask = None
        n_annotators = 0

        for mdir in mask_dirs:
            mask_file = mdir / mid_img_path.name
            if mask_file.exists():
                m = np.array(Image.open(mask_file).convert('L'))
                if m.shape != img.shape:
                    continue
                area = int((m > 0).sum())
                mask_areas.append(area)
                if combined_mask is None:
                    combined_mask = (m > 0).astype(np.float32)
                else:
                    combined_mask += (m > 0).astype(np.float32)
                n_annotators += 1

        if n_annotators == 0 or combined_mask is None:
            continue

        # Konsensus maske (>=50% annotator uyumu)
        consensus = (combined_mask / n_annotators) >= 0.5
        consensus_area = int(consensus.sum())
        if consensus_area < 4:  # Cok kucuk maskeler atla
            continue

        # Boyut ozellikleri
        diameter_px = float(np.sqrt(consensus_area / np.pi) * 2)

        # Morfoloji
        morph = compute_morphology_features(consensus)

        # Yogunluk
        intensity = compute_intensity_features(img, consensus)

        # Kenar keskinligi
        margin = compute_margin_sharpness(img, consensus)

        # GLCM doku
        glcm = compute_glcm_features(img, consensus)

        # Annotator uyumu: varyans
        annotator_area_std = float(np.std(mask_areas)) if len(mask_areas) > 1 else 0.0

        record = {
            'patient_id': pdir.name,
            'nodule': ndir.name,
            'image_path': str(mid_img_path),
            'mask_dirs': [str(m) for m in mask_dirs],
            # Boyut
            'diameter_px': diameter_px,
            'consensus_area': float(consensus_area),
            # Morfoloji
            'compactness': morph['compactness'],
            'eccentricity': morph['eccentricity'],
            'solidity': morph['solidity'],
            'perimeter': morph['perimeter'],
            'margin_sharpness': margin,
            # Yogunluk
            'intensity_mean': intensity['intensity_mean'],
            'intensity_std': intensity['intensity_std'],
            'surround_mean': intensity['surround_mean'],
            'surround_std': intensity['surround_std'],
            'contrast_ratio': intensity['contrast_ratio'],
            # GLCM
            'glcm_homogeneity': glcm['glcm_homogeneity'],
            'glcm_contrast': glcm['glcm_contrast'],
            'glcm_energy': glcm['glcm_energy'],
            'glcm_correlation': glcm['glcm_correlation'],
            # Meta
            'n_annotators': n_annotators,
            'n_slices': len(img_files),
            'annotator_area_std': annotator_area_std,
        }
        nodule_data.append(record)

    if (pi + 1) % 200 == 0:
        print(f"  {pi+1}/{len(patient_dirs)} hasta islendi, "
              f"{len(nodule_data)} nodul, {time.time()-t0:.0f}s")

df = pd.DataFrame(nodule_data)
elapsed = time.time() - t0
print(f"\nOzellik cikarimi tamamlandi: {elapsed:.0f}s")
print(f"Toplam nodul: {len(df)}")
print(f"Toplam hasta: {df['patient_id'].nunique()}")
print(f"\n--- Ozellik Istatistikleri ---")
feature_cols = [
    'diameter_px', 'consensus_area', 'compactness', 'eccentricity',
    'solidity', 'margin_sharpness', 'intensity_mean', 'intensity_std',
    'contrast_ratio', 'glcm_homogeneity', 'glcm_contrast',
    'glcm_energy', 'glcm_correlation', 'n_annotators', 'n_slices'
]
print(df[feature_cols].describe().round(3).to_string())

---
## 3. Malignite Proxy Etiketleme ve Dagilim

In [ ]:
def compute_malignancy_proxy(df):
    """Multi-feature malignite proxy skoru (1-5 skala).
    
    Her ozellik [0, 1] araligina normalize edilir,
    agirlikli toplam -> 1-5 skalaya eslenir.
    """
    scores = pd.DataFrame(index=df.index)
    
    def norm_01(s):
        smin, smax = s.min(), s.max()
        if smax - smin < 1e-8:
            return pd.Series(0.5, index=s.index)
        return (s - smin) / (smax - smin)
    
    scores['size'] = norm_01(df['diameter_px'])
    scores['irregularity'] = 1.0 - norm_01(df['compactness'])
    scores['spiculation'] = 1.0 - norm_01(df['solidity'])
    scores['elongation'] = norm_01(df['eccentricity'])
    scores['margin_blur'] = 1.0 - norm_01(df['margin_sharpness'])
    scores['contrast'] = norm_01(df['contrast_ratio'])
    scores['heterogeneity'] = norm_01(df['intensity_std'])
    scores['texture_irregular'] = 1.0 - norm_01(df['glcm_homogeneity'])
    scores['texture_contrast'] = norm_01(df['glcm_contrast'])
    
    weights = {
        'size': 0.25,
        'irregularity': 0.15,
        'spiculation': 0.12,
        'elongation': 0.05,
        'margin_blur': 0.12,
        'contrast': 0.08,
        'heterogeneity': 0.08,
        'texture_irregular': 0.08,
        'texture_contrast': 0.07,
    }
    
    raw_score = sum(scores[k] * w for k, w in weights.items())
    malignancy_score = 1.0 + 4.0 * raw_score
    
    return malignancy_score, scores


df['malignancy_score'], sub_scores = compute_malignancy_proxy(df)

# 5-sinif malignite: eşit aralıklı bins
df['malignancy_class'] = pd.cut(
    df['malignancy_score'],
    bins=[0.0, 1.8, 2.6, 3.4, 4.2, 6.0],
    labels=[0, 1, 2, 3, 4]
).astype(int)

# FIX: Mutlak eşik (3.5) yerine yüzde 75. persentil kullan.
# Proxy skor LIDC-IDRI veri setinde [1.5, 3.3] arasında kalır (hiç 3.5'i geçmez).
# Persentil tabanlı eşik her zaman ~%25 malign örnek üretir.
MALIGN_THRESHOLD = df['malignancy_score'].quantile(0.75)
df['malignant'] = (df['malignancy_score'] >= MALIGN_THRESHOLD).astype(int)

print("--- Malignite Proxy Skor Dagilimi ---")
print(df['malignancy_score'].describe().round(3))
print(f"\nMalign esigi (75. persentil): {MALIGN_THRESHOLD:.4f}")
print(f"\n--- Binary Dagilim ---")
print(f"Benign (0): {(df['malignant'] == 0).sum()}")
print(f"Malign (1): {(df['malignant'] == 1).sum()}")
print(f"Malign oran: {df['malignant'].mean():.2%}")
print(f"\n--- 5-Sinif Dagilim ---")
for c in range(5):
    n = (df['malignancy_class'] == c).sum()
    pct = n / len(df) * 100
    labels = ['Kesinlikle Benign', 'Muhtemelen Benign',
              'Belirsiz', 'Muhtemelen Malign', 'Kesinlikle Malign']
    print(f"  Sinif {c} ({labels[c]}): {n:>5d} ({pct:.1f}%)")

print(f"\n--- Sub-skor istatistikleri ---")
print(sub_scores.describe().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Malignite skor dagilimi
axes[0, 0].hist(df['malignancy_score'], bins=40, edgecolor='black',
                alpha=0.7, color='steelblue')
axes[0, 0].axvline(x=3.5, color='red', linestyle='--',
                   label='Malign esigi (3.5)', linewidth=2)
axes[0, 0].set_title('Malignite Proxy Skor Dagilimi', fontweight='bold')
axes[0, 0].set_xlabel('Malignite Skoru (1-5)')
axes[0, 0].set_ylabel('Sayi')
axes[0, 0].legend()

# 2. Binary sinif dengesi
binary_counts = df['malignant'].value_counts().sort_index()
colors_bin = ['#2ecc71', '#e74c3c']
axes[0, 1].bar(['Benign', 'Malign'], binary_counts.values,
               color=colors_bin, edgecolor='black')
axes[0, 1].set_title('Binary Sinif Dengesi', fontweight='bold')
axes[0, 1].set_ylabel('Sayi')
for i, v in enumerate(binary_counts.values):
    axes[0, 1].text(i, v + 5, str(v), ha='center', fontweight='bold')

# 3. 5-sinif dagilimi
class_counts = df['malignancy_class'].value_counts().sort_index()
colors_5 = ['#27ae60', '#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
class_labels = ['1-KB', '2-MB', '3-Bel', '4-MM', '5-KM']
axes[0, 2].bar(class_labels,
               [class_counts.get(i, 0) for i in range(5)],
               color=colors_5, edgecolor='black')
axes[0, 2].set_title('5-Sinif Malignite Dagilimi', fontweight='bold')
axes[0, 2].set_ylabel('Sayi')

# 4. Ozellik korelasyonu (malignite ile)
feature_corr_cols = [
    'diameter_px', 'compactness', 'solidity', 'eccentricity',
    'margin_sharpness', 'contrast_ratio', 'intensity_std',
    'glcm_homogeneity', 'glcm_contrast', 'n_annotators'
]
correlations = df[feature_corr_cols + ['malignancy_score']].corr()['malignancy_score'].drop('malignancy_score')
correlations = correlations.sort_values()
colors_corr = ['#e74c3c' if v < 0 else '#2ecc71' for v in correlations]
axes[1, 0].barh(correlations.index, correlations.values,
                color=colors_corr, edgecolor='black', alpha=0.8)
axes[1, 0].set_title('Ozellik - Malignite Korelasyonu', fontweight='bold')
axes[1, 0].set_xlabel('Pearson r')
axes[1, 0].axvline(x=0, color='black', linewidth=0.5)

# 5. Diameter vs Malignite
axes[1, 1].scatter(
    df['diameter_px'], df['malignancy_score'],
    c=df['malignant'], cmap='RdYlGn_r', alpha=0.5, s=15
)
axes[1, 1].set_title('Cap vs Malignite Skoru', fontweight='bold')
axes[1, 1].set_xlabel('Cap (piksel)')
axes[1, 1].set_ylabel('Malignite Skoru')
axes[1, 1].axhline(y=3.5, color='red', linestyle='--', alpha=0.5)

# 6. Annotator sayisi vs malignite
ann_groups = df.groupby('n_annotators')['malignancy_score'].mean()
axes[1, 2].bar(ann_groups.index, ann_groups.values,
               color='coral', edgecolor='black')
axes[1, 2].set_title('Annotator Sayisi vs Ort. Malignite', fontweight='bold')
axes[1, 2].set_xlabel('Annotator Sayisi')
axes[1, 2].set_ylabel('Ort. Malignite Skoru')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'malignancy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dagilim grafigi kaydedildi.")

---
## 4. Egitim/Dogrulama Bolmesi ve Dataset

In [ ]:
# ---------------------------------------------------------------
# Hasta bazli bolme (veri sizintisi onleme)
# ---------------------------------------------------------------
patients = df['patient_id'].unique()
np.random.shuffle(patients)
split_idx = int(len(patients) * 0.8)
train_patients = set(patients[:split_idx])
val_patients = set(patients[split_idx:])

train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)

print(f"Train: {len(train_df)} nodul ({len(train_patients)} hasta)")
print(f"Val:   {len(val_df)} nodul ({len(val_patients)} hasta)")
print(f"Train malign oran: {train_df['malignant'].mean():.2%}")
print(f"Val   malign oran: {val_df['malignant'].mean():.2%}")

# ---------------------------------------------------------------
# Tabular ozellikleri normalize et
# ---------------------------------------------------------------
TABULAR_FEATURES = [
    'diameter_px', 'consensus_area', 'compactness', 'eccentricity',
    'solidity', 'perimeter', 'margin_sharpness',
    'intensity_mean', 'intensity_std', 'surround_mean', 'surround_std',
    'contrast_ratio', 'glcm_homogeneity', 'glcm_contrast',
    'glcm_energy', 'glcm_correlation',
    'n_annotators', 'n_slices', 'annotator_area_std',
]
N_TAB_FEATURES = len(TABULAR_FEATURES)
print(f"Tabular ozellik sayisi: {N_TAB_FEATURES}")

scaler = StandardScaler()
train_tab = scaler.fit_transform(train_df[TABULAR_FEATURES].values.astype(np.float32))
val_tab = scaler.transform(val_df[TABULAR_FEATURES].values.astype(np.float32))

# NaN/Inf kontrolu
train_tab = np.nan_to_num(train_tab, nan=0.0, posinf=0.0, neginf=0.0)
val_tab = np.nan_to_num(val_tab, nan=0.0, posinf=0.0, neginf=0.0)

train_df['_tab_idx'] = range(len(train_df))
val_df['_tab_idx'] = range(len(val_df))


# ---------------------------------------------------------------
# Dataset sinifi
# ---------------------------------------------------------------
class MalignancyDataset(Dataset):
    """Nodul malignite dataset -- goruntu + tabular ozellikler.
    
    Returns:
        image_tensor: (3, H, W) normalize edilmis goruntu
        tabular_tensor: (N_TAB,) normalize edilmis tabular ozellikler
        binary_label: 0 veya 1
        malignancy_class: 0-4 (5 sinif)
    """
    def __init__(self, dataframe, tabular_array, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.tab = tabular_array
        self.is_train = is_train
        # ImageNet normalization
        self.mean = np.array([0.485, 0.456, 0.406])
        self.std = np.array([0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def _load_and_crop(self, row):
        """Goruntu yukle ve nodul bolgesine crop yap."""
        img = np.array(Image.open(row['image_path']).convert('L'))
        mask_dirs_list = row['mask_dirs']

        # Konsensus maske olustur
        combined = np.zeros(img.shape, dtype=np.float32)
        n = 0
        for mdir in mask_dirs_list:
            mask_file = Path(mdir) / Path(row['image_path']).name
            if mask_file.exists():
                m = np.array(Image.open(mask_file).convert('L'))
                if m.shape == img.shape:
                    combined += (m > 0).astype(np.float32)
                    n += 1
        if n > 0:
            consensus = (combined / n) >= 0.5
            if consensus.any():
                rows_w = np.where(np.any(consensus, axis=1))[0]
                cols_w = np.where(np.any(consensus, axis=0))[0]
                cy = (rows_w[0] + rows_w[-1]) // 2
                cx = (cols_w[0] + cols_w[-1]) // 2
                h = rows_w[-1] - rows_w[0]
                w = cols_w[-1] - cols_w[0]
                margin = max(h, w)
                half = max(margin, 32)
                y1 = max(0, cy - half)
                y2 = min(img.shape[0], cy + half)
                x1 = max(0, cx - half)
                x2 = min(img.shape[1], cx + half)
                img = img[y1:y2, x1:x2]

        if img.size == 0 or img.shape[0] < 4 or img.shape[1] < 4:
            img = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)
        return img

    def _augment(self, img):
        """Basit augmentasyon (train icin)."""
        if np.random.random() < 0.5:
            img = np.fliplr(img).copy()
        if np.random.random() < 0.5:
            img = np.flipud(img).copy()
        if np.random.random() < 0.5:
            k = np.random.randint(1, 4)
            img = np.rot90(img, k).copy()
        # Parlaklik jitter
        if np.random.random() < 0.3:
            factor = np.random.uniform(0.8, 1.2)
            img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_and_crop(row)

        if self.is_train:
            img = self._augment(img)

        # Resize
        img_pil = Image.fromarray(img).resize(
            (PATCH_SIZE, PATCH_SIZE), Image.BILINEAR
        )
        img_np = np.array(img_pil, dtype=np.float32) / 255.0

        # 3 kanal + normalize
        img_rgb = np.stack([img_np, img_np, img_np], axis=-1)
        img_rgb = (img_rgb - self.mean) / self.std
        img_tensor = torch.from_numpy(
            img_rgb.transpose(2, 0, 1).astype(np.float32)
        )

        # Tabular
        tab_idx = row['_tab_idx']
        tab_tensor = torch.from_numpy(
            self.tab[tab_idx].astype(np.float32)
        )

        # Etiketler
        binary_label = torch.tensor(
            row['malignant'], dtype=torch.float32
        )
        malignancy_class = torch.tensor(
            row['malignancy_class'], dtype=torch.long
        )

        return img_tensor, tab_tensor, binary_label, malignancy_class


train_dataset = MalignancyDataset(train_df, train_tab, is_train=True)
val_dataset = MalignancyDataset(val_df, val_tab, is_train=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"Train dataset: {len(train_dataset)}, Val dataset: {len(val_dataset)}")
img_s, tab_s, bin_s, cls_s = train_dataset[0]
print(f"Goruntu: {img_s.shape}, Tabular: {tab_s.shape}, "
      f"Binary: {bin_s.item()}, Class: {cls_s.item()}")

---
## 5. Model -- EfficientNet-B0 + Tabular Fusion

In [ ]:
class MalignancyNet(nn.Module):
    """EfficientNet-B0 + Tabular Feature Fusion.

    Image branch: EfficientNet-B0 -> 1280-dim features
    Tabular branch: FC layers -> 128-dim  (64 → 128 arttirildi)
    Fusion: concat -> FC -> outputs
    """
    def __init__(self, n_tabular_features, n_malignancy_classes=5):
        super().__init__()
        # Image branch -- EfficientNet-B0
        effnet = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
        )
        self.image_features = effnet.features
        self.image_pool = nn.AdaptiveAvgPool2d(1)

        # Tabular branch -- 64 → 128 (daha fazla kapasite)
        self.tabular_branch = nn.Sequential(
            nn.Linear(n_tabular_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 128),
            nn.ReLU(),
        )

        # Fusion head
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1280 + 128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.fc_binary = nn.Linear(256, 1)
        self.fc_malignancy = nn.Linear(256, n_malignancy_classes)

    def forward(self, image, tabular):
        img_feat = self.image_features(image)
        img_feat = self.image_pool(img_feat).flatten(1)   # (B, 1280)
        tab_feat = self.tabular_branch(tabular)            # (B, 128)
        fused = torch.cat([img_feat, tab_feat], dim=1)    # (B, 1408)
        fused = self.classifier(fused)
        binary_out = self.fc_binary(fused).squeeze(-1)
        malignancy_out = self.fc_malignancy(fused)
        return binary_out, malignancy_out


model = MalignancyNet(
    n_tabular_features=N_TAB_FEATURES,
    n_malignancy_classes=N_MALIGNANCY_CLASSES
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: EfficientNet-B0 + Tabular Fusion (Gelismis)")
print(f"Toplam parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"Trainable: {n_trainable:,}")
print(f"Tabular branch: {N_TAB_FEATURES} -> 128-dim (onceki: 64-dim)")

---
## 6. Loss ve Optimizer

In [ ]:
class FocalLoss(nn.Module):
    """Binary Focal Loss."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()


class MulticlassFocalLoss(nn.Module):
    """Multiclass Focal Loss — class collapse'i onler."""
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, pred, target):
        # Label smoothing ile CE
        ce = F.cross_entropy(pred, target, weight=self.weight,
                             label_smoothing=self.label_smoothing,
                             reduction='none')
        pt = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        return focal.mean()


# --- Binary loss: Focal ---
criterion_binary = FocalLoss(alpha=0.25, gamma=2.0)

# --- 5-sinif loss: Multiclass Focal (CE yerine) ---
# FIX: Her zaman 5 eleman, eksik siniflar icin count=1
class_counts_series = train_df['malignancy_class'].value_counts()
class_counts_arr = np.array([
    class_counts_series.get(i, 0) for i in range(N_MALIGNANCY_CLASSES)
], dtype=np.float32)
class_counts_arr = np.maximum(class_counts_arr, 1)
class_weights_raw = 1.0 / class_counts_arr
class_weights_raw = class_weights_raw / class_weights_raw.sum() * N_MALIGNANCY_CLASSES
# Agirliklari kliple (cok yuksek agirliklar loss patlamasina sebep oluyordu)
class_weights_raw = np.clip(class_weights_raw, 0.1, 5.0)
class_weights_tensor = torch.tensor(class_weights_raw, dtype=torch.float32).to(device)

criterion_malignancy = MulticlassFocalLoss(
    gamma=2.0, weight=class_weights_tensor, label_smoothing=0.1
)

print(f"5-sinif agirliklari (klipli): {class_weights_raw.round(3)}")
print(f"Train sinif dagilimi: {class_counts_series.sort_index().to_dict()}")

# --- Optimizer & Scheduler ---
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
# Warm restarts: her 20 epochta LR sifirlanir, yeni minimumlar aranir
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=1e-6
)

# Loss agirliklari -- 5-class'i hafiflet (eskiden 0.5, loss ~19 oluyordu)
BINARY_WEIGHT = 1.0
MALIGNANCY_WEIGHT = 0.2   # 0.5 → 0.2

print(f"\nBinary loss:   Focal (alpha=0.25, gamma=2)")
print(f"5-sinif loss:  Multiclass Focal (gamma=2, label_smooth=0.1, weight klipli)")
print(f"Optimizer:     AdamW (lr=2e-4, wd=1e-4)")
print(f"Scheduler:     CosineAnnealingWarmRestarts (T_0=20)")
print(f"Loss agirlik:  binary={BINARY_WEIGHT}, malignancy={MALIGNANCY_WEIGHT}")

---
## 7. Egitim

In [ ]:
best_val_auc = 0.0
best_cls_acc = 0.0
history = {
    'train_loss': [], 'val_loss': [],
    'val_binary_auc': [], 'val_5class_acc': [], 'lr': []
}
amp_scaler = torch.cuda.amp.GradScaler()

print(f"Egitim basliyor -- {NUM_EPOCHS} epoch")
print(f"Loss agirliklari: binary={BINARY_WEIGHT}, malignancy={MALIGNANCY_WEIGHT}")
print("=" * 75)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # --- TRAIN ---
    model.train()
    train_loss = 0.0
    n_train = 0

    for images, tabular, bin_labels, cls_labels in train_loader:
        images = images.to(device)
        tabular = tabular.to(device)
        bin_labels = bin_labels.to(device)
        cls_labels = cls_labels.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            bin_pred, cls_pred = model(images, tabular)
            loss_bin = criterion_binary(bin_pred, bin_labels)
            loss_cls = criterion_malignancy(cls_pred, cls_labels)
            loss = BINARY_WEIGHT * loss_bin + MALIGNANCY_WEIGHT * loss_cls

        amp_scaler.scale(loss).backward()
        amp_scaler.step(optimizer)
        amp_scaler.update()

        train_loss += loss.item() * images.size(0)
        n_train += images.size(0)

    train_loss /= max(n_train, 1)

    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    all_bin_preds = []
    all_bin_labels = []
    all_cls_preds = []
    all_cls_labels = []
    n_val = 0

    with torch.no_grad():
        for images, tabular, bin_labels, cls_labels in val_loader:
            images = images.to(device)
            tabular = tabular.to(device)
            bin_labels = bin_labels.to(device)
            cls_labels = cls_labels.to(device)

            with torch.cuda.amp.autocast():
                bin_pred, cls_pred = model(images, tabular)
                loss_bin = criterion_binary(bin_pred, bin_labels)
                loss_cls = criterion_malignancy(cls_pred, cls_labels)
                loss = BINARY_WEIGHT * loss_bin + MALIGNANCY_WEIGHT * loss_cls

            val_loss += loss.item() * images.size(0)
            all_bin_preds.extend(torch.sigmoid(bin_pred).cpu().numpy())
            all_bin_labels.extend(bin_labels.cpu().numpy())
            all_cls_preds.extend(cls_pred.argmax(dim=1).cpu().numpy())
            all_cls_labels.extend(cls_labels.cpu().numpy())
            n_val += images.size(0)

    val_loss /= max(n_val, 1)

    all_bin_preds = np.array(all_bin_preds)
    all_bin_labels = np.array(all_bin_labels)
    all_cls_preds = np.array(all_cls_preds)
    all_cls_labels = np.array(all_cls_labels)

    if len(np.unique(all_bin_labels)) > 1:
        val_auc = roc_auc_score(all_bin_labels, all_bin_preds)
    else:
        val_auc = 0.0

    cls_acc = float((all_cls_preds == all_cls_labels).mean())

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_binary_auc'].append(val_auc)
    history['val_5class_acc'].append(cls_acc)
    history['lr'].append(current_lr)

    # Best model — AUC iyileştiyse kaydet
    save_checkpoint = val_auc > best_val_auc
    if save_checkpoint:
        best_val_auc = val_auc
        best_cls_acc = cls_acc
        marker = ' << BEST'
    else:
        marker = ''

    # FIX: Son epoch'ta her zaman kaydet (AUC 0 olsa bile)
    if save_checkpoint or epoch == NUM_EPOCHS - 1:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_binary_auc': float(val_auc),
            'val_5class_acc': float(cls_acc),
            'n_tabular_features': N_TAB_FEATURES,
        }, OUTPUT_DIR / 'malignancy_best_model.pth')

    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        print(
            f"Epoch {epoch+1:>3d}/{NUM_EPOCHS} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"AUC: {val_auc:.4f} | "
            f"5cls-Acc: {cls_acc:.4f} | "
            f"LR: {current_lr:.6f} | "
            f"{elapsed/60:.1f}m{marker}"
        )

total_time = time.time() - start_time
print(f"\nEgitim tamamlandi! Sure: {total_time/60:.1f} dk")
print(f"En iyi Binary AUC: {best_val_auc:.4f}")
print(f"En iyi 5-sinif Acc: {best_cls_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_title('Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Binary AUC
axes[0, 1].plot(history['val_binary_auc'], linewidth=2, color='#e74c3c')
axes[0, 1].axhline(y=0.85, color='green', linestyle='--',
                   label='Hedef AUC=0.85', alpha=0.6)
axes[0, 1].set_title('Binary Malignite AUC', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('AUC')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0, 1)

# 5-class Accuracy
axes[1, 0].plot(history['val_5class_acc'], linewidth=2, color='#3498db')
axes[1, 0].axhline(y=0.55, color='green', linestyle='--',
                   label='Hedef Acc=0.55', alpha=0.6)
axes[1, 0].set_title('5-Sinif Malignite Accuracy', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim(0, 1)

# Learning Rate
axes[1, 1].plot(history['lr'], linewidth=2, color='#2ecc71')
axes[1, 1].set_title('Learning Rate (Cosine)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('LR')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'malignancy_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Egitim grafikleri kaydedildi.")

In [ ]:
# ---------------------------------------------------------------
# En iyi modeli yukle ve detayli degerlendirme
# ---------------------------------------------------------------
checkpoint = torch.load(
    OUTPUT_DIR / 'malignancy_best_model.pth',
    map_location=device, weights_only=False
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Best model yuklendi "
      f"(Epoch {checkpoint['epoch']+1}, "
      f"AUC={checkpoint['val_binary_auc']:.4f})")

# Tam validasyon tahminleri
all_bin_preds = []
all_bin_labels = []
all_cls_preds = []
all_cls_labels = []
all_cls_probs = []

with torch.no_grad():
    for images, tabular, bin_labels, cls_labels in val_loader:
        images = images.to(device)
        tabular = tabular.to(device)
        with torch.cuda.amp.autocast():
            bin_pred, cls_pred = model(images, tabular)
        all_bin_preds.extend(torch.sigmoid(bin_pred).cpu().numpy())
        all_bin_labels.extend(bin_labels.numpy())
        all_cls_preds.extend(cls_pred.argmax(dim=1).cpu().numpy())
        all_cls_labels.extend(cls_labels.numpy())
        all_cls_probs.extend(
            F.softmax(cls_pred, dim=1).cpu().numpy()
        )

all_bin_preds = np.array(all_bin_preds)
all_bin_labels = np.array(all_bin_labels)
all_cls_preds = np.array(all_cls_preds)
all_cls_labels = np.array(all_cls_labels)
all_cls_probs = np.array(all_cls_probs)

# --- Gorsellestirme ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. ROC egrisi
if len(np.unique(all_bin_labels)) > 1:
    fpr, tpr, thresholds = roc_curve(all_bin_labels, all_bin_preds)
    auc_val = roc_auc_score(all_bin_labels, all_bin_preds)
    axes[0].plot(fpr, tpr, linewidth=2, color='#e74c3c',
                label=f'AUC = {auc_val:.4f}')
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[0].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Egrisi -- Binary Malignite', fontweight='bold')
    axes[0].legend(fontsize=12)
    axes[0].grid(True, alpha=0.2)
else:
    auc_val = 0.0
    axes[0].text(0.5, 0.5, 'Tek sinif -- ROC hesaplanamadi',
                ha='center', va='center',
                transform=axes[0].transAxes, fontsize=12)
    axes[0].set_title('ROC Egrisi', fontweight='bold')

# 2. Binary confusion matrix
bin_preds_discrete = (all_bin_preds >= 0.5).astype(int)
cm_bin = confusion_matrix(all_bin_labels, bin_preds_discrete)
im1 = axes[1].imshow(cm_bin, interpolation='nearest', cmap='Reds')
axes[1].set_title('Binary Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Tahmin')
axes[1].set_ylabel('Gercek')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Benign', 'Malign'])
axes[1].set_yticklabels(['Benign', 'Malign'])
for i in range(cm_bin.shape[0]):
    for j in range(cm_bin.shape[1]):
        axes[1].text(j, i, str(cm_bin[i, j]), ha='center', va='center',
                    color='white' if cm_bin[i, j] > cm_bin.max() / 2 else 'black',
                    fontsize=14, fontweight='bold')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# 3. 5-sinif confusion matrix
# FIX: labels ve target_names'i mevcut siniflara gore hizala
cls_names_all = ['1-KB', '2-MB', '3-Bel', '4-MM', '5-KM']
present_labels = sorted(set(all_cls_labels.tolist()) | set(all_cls_preds.tolist()))
present_names = [cls_names_all[i] for i in present_labels if i < len(cls_names_all)]

cm_cls = confusion_matrix(all_cls_labels, all_cls_preds, labels=present_labels)
im2 = axes[2].imshow(cm_cls, interpolation='nearest', cmap='Blues')
axes[2].set_title('5-Sinif Confusion Matrix', fontweight='bold')
axes[2].set_xlabel('Tahmin')
axes[2].set_ylabel('Gercek')
axes[2].set_xticks(range(len(present_labels)))
axes[2].set_yticks(range(len(present_labels)))
axes[2].set_xticklabels(present_names, fontsize=8)
axes[2].set_yticklabels(present_names, fontsize=8)
for i in range(cm_cls.shape[0]):
    for j in range(cm_cls.shape[1]):
        axes[2].text(j, i, str(cm_cls[i, j]), ha='center', va='center',
                    color='white' if cm_cls[i, j] > cm_cls.max() / 2 else 'black',
                    fontsize=11, fontweight='bold')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'malignancy_evaluation.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Classification reports
print("\n--- Binary Classification Report ---")
print(classification_report(
    all_bin_labels, bin_preds_discrete,
    target_names=['Benign', 'Malign'], zero_division=0
))

print("--- 5-Sinif Classification Report ---")
# FIX: labels parametresi ile mevcut siniflari belirt
print(classification_report(
    all_cls_labels, all_cls_preds,
    labels=present_labels,
    target_names=present_names,
    zero_division=0
))

# Feature importance -- tabular ozelliklerin tahminle korelasyonu
print("\n--- Ozellik Onemi (tahminle korelasyon) ---")
val_tab_df = pd.DataFrame(val_tab, columns=TABULAR_FEATURES)
val_tab_df['bin_pred'] = all_bin_preds
feat_corr = val_tab_df.corr()['bin_pred'].drop('bin_pred').abs().sort_values(ascending=False)
for feat, corr_val in feat_corr.items():
    print(f"  {feat:<25s} {corr_val:.4f}")

In [ ]:
# ---------------------------------------------------------------
# Sonuclari Kaydet
# ---------------------------------------------------------------
import json

# --- 1. Tahminler CSV ---
results_df = val_df.copy()
results_df['bin_pred'] = all_bin_preds
results_df['bin_pred_label'] = bin_preds_discrete
results_df['cls_pred'] = all_cls_preds
# Olasilik kolonlari
cls_labels_full = [
    'prob_certainly_benign', 'prob_probably_benign',
    'prob_indeterminate', 'prob_probably_malignant',
    'prob_certainly_malignant'
]
for i, name in enumerate(cls_labels_full):
    if i < all_cls_probs.shape[1]:
        results_df[name] = all_cls_probs[:, i]

# mask_dirs -> string
results_df['mask_dirs'] = results_df['mask_dirs'].apply(str)
results_df.to_csv(OUTPUT_DIR / 'malignancy_predictions.csv', index=False)
print(f"Tahminler kaydedildi: malignancy_predictions.csv")

# --- 2. Metrikler CSV ---
final_auc = float(auc_val)
final_cls_acc = float((all_cls_preds == all_cls_labels).mean())

metrics = {
    'binary_auc': final_auc,
    '5class_accuracy': final_cls_acc,
    'n_val_samples': int(len(all_bin_labels)),
    'n_train_samples': int(len(train_df)),
    'best_epoch': int(checkpoint['epoch'] + 1),
    'total_epochs': int(NUM_EPOCHS),
    'n_features': int(N_TAB_FEATURES),
}
pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / 'malignancy_metrics.csv', index=False
)
print(f"Metrikler kaydedildi: malignancy_metrics.csv")

# --- 3. Training history ---
pd.DataFrame(history).to_csv(
    OUTPUT_DIR / 'malignancy_training_history.csv', index=False
)
print(f"Egitim historisi kaydedildi: malignancy_training_history.csv")

# --- 4. Pipeline config JSON ---
# Tum numpy degerlerini Python native tiplere cevir
pipeline_config = {
    'model': {
        'architecture': 'EfficientNet-B0 + Tabular Fusion',
        'image_branch': 'EfficientNet-B0 (ImageNet pretrained)',
        'tabular_branch': f'{int(N_TAB_FEATURES)} -> 64',
        'fusion_dim': int(1280 + 64),
        'n_params': int(n_params),
        'patch_size': int(PATCH_SIZE),
        'n_tabular_features': int(N_TAB_FEATURES),
        'tabular_features': TABULAR_FEATURES,
    },
    'training': {
        'epochs': int(NUM_EPOCHS),
        'batch_size': int(BATCH_SIZE),
        'optimizer': 'AdamW',
        'lr': float(1e-4),
        'weight_decay': float(1e-4),
        'scheduler': 'CosineAnnealingLR',
        'binary_loss': 'FocalLoss (alpha=0.25, gamma=2)',
        'multiclass_loss': 'LabelSmoothingCE (smoothing=0.1)',
        'amp': True,
    },
    'dataset': {
        'source': 'zhangweiled/lidcidri (LIDC-IDRI slices)',
        'n_train': int(len(train_df)),
        'n_val': int(len(val_df)),
        'n_patients_train': int(len(train_patients)),
        'n_patients_val': int(len(val_patients)),
        'split': '80/20 patient-based',
    },
    'results': {
        'binary_auc': float(final_auc),
        '5class_accuracy': float(final_cls_acc),
        'best_epoch': int(checkpoint['epoch'] + 1),
    },
    'comparison_nb07': {
        'nb07_approach': 'Size-only proxy (diameter-based 4-class)',
        'nb07_model': 'ResNet-50 + CBAM',
        'nb13_approach': 'Multi-feature proxy (size+morph+intensity+GLCM, 5-class)',
        'nb13_model': 'EfficientNet-B0 + Tabular Fusion',
        'improvements': [
            '15 radiomic features vs size-only',
            'Image + tabular fusion vs image-only',
            '5-class ordinal vs 4-class categorical',
            'Lighter model (EfficientNet-B0 vs ResNet-50)',
            'Label smoothing for ordinal classes',
        ],
    },
    'scaler': {
        'mean': [float(x) for x in scaler.mean_],
        'scale': [float(x) for x in scaler.scale_],
    },
    'malignancy_proxy': {
        'method': 'Weighted multi-feature scoring',
        'features_used': [
            'size (0.25)', 'irregularity (0.15)',
            'spiculation (0.12)', 'elongation (0.05)',
            'margin_blur (0.12)', 'contrast (0.08)',
            'heterogeneity (0.08)', 'texture_irregular (0.08)',
            'texture_contrast (0.07)',
        ],
        'binary_threshold': float(3.5),
        'class_bins': '[0, 1.8, 2.6, 3.4, 4.2, 6.0]',
    },
}

with open(OUTPUT_DIR / 'pipeline_config.json', 'w') as f:
    json.dump(pipeline_config, f, indent=2, ensure_ascii=False)
print(f"Pipeline config kaydedildi: pipeline_config.json")

# --- Kayit ozeti ---
print(f"\nKaydedilen dosyalar:")
for f_path in sorted(OUTPUT_DIR.glob('malignancy_*')):
    size_kb = f_path.stat().st_size / 1024
    if size_kb > 1024:
        print(f"  {f_path.name} ({size_kb/1024:.1f} MB)")
    else:
        print(f"  {f_path.name} ({size_kb:.0f} KB)")
for f_path in sorted(OUTPUT_DIR.glob('pipeline_*')):
    size_kb = f_path.stat().st_size / 1024
    print(f"  {f_path.name} ({size_kb:.0f} KB)")

In [ ]:
print("\n" + "=" * 70)
print("AlpCAN BT-3: Nodul Malignite Siniflandirmasi -- OZET")
print("=" * 70)

print(f"\n--- Dataset ---")
print(f"  Kaynak: zhangweiled/lidcidri (LIDC-IDRI slices)")
print(f"  Train: {len(train_df)} nodul ({len(train_patients)} hasta)")
print(f"  Val:   {len(val_df)} nodul ({len(val_patients)} hasta)")
print(f"  Etiket: Multi-feature malignite proxy (5 sinif, 1-5 skala)")
print(f"  Proxy ozellikleri: boyut, morfoloji, yogunluk, GLCM doku")

print(f"\n--- Model ---")
print(f"  Mimari: EfficientNet-B0 + Tabular Feature Fusion")
print(f"  Image: {PATCH_SIZE}x{PATCH_SIZE}x3 -> 1280-dim")
print(f"  Tabular: {N_TAB_FEATURES} ozellik -> 64-dim")
print(f"  Fusion: 1344 -> 256 -> binary + 5-class")
print(f"  Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Loss: Focal (binary) + LabelSmoothing CE (5-class)")
print(f"  Optimizer: AdamW (lr=1e-4)")
print(f"  Epoch: {NUM_EPOCHS}")

print(f"\n--- Performans ---")
print(f"  Binary AUC:      {final_auc:.4f}  (hedef > 0.85)")
print(f"  5-sinif Acc:     {final_cls_acc:.4f}  (hedef > 0.55)")
print(f"  Best Epoch:      {int(checkpoint['epoch'] + 1)}")
print(f"  Egitim suresi:   {total_time/60:.1f} dk")

print(f"\n--- NB07 Karsilastirma ---")
print(f"  NB07: Boyut-only proxy, ResNet-50+CBAM, 4-sinif risk")
print(f"  NB13: Multi-feature proxy, EfficientNet-B0+Tabular, 5-sinif malignite")
print(f"  Iyilestirmeler:")
print(f"    - 15 radyomik ozellik vs sadece boyut")
print(f"    - Goruntu + tabular fusion vs sadece goruntu")
print(f"    - 5-sinif ordinal vs 4-sinif kategorik")
print(f"    - Daha hafif model (EfficientNet-B0 vs ResNet-50)")

print(f"\n--- Cikti Dosyalari ---")
print(f"  malignancy_best_model.pth      -- Model agirliklari")
print(f"  malignancy_metrics.csv          -- Final metrikler")
print(f"  malignancy_training_history.csv -- Epoch historisi")
print(f"  malignancy_training_curves.png  -- Egitim grafikleri")
print(f"  malignancy_evaluation.png       -- ROC + CM")
print(f"  malignancy_distribution.png     -- Dagilim grafikleri")
print(f"  malignancy_predictions.csv      -- Val tahminleri")
print(f"  pipeline_config.json            -- Pipeline konfigurasyonu")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Gercek LIDC-IDRI XML annotasyonlari ile yeniden egitim")
print(f"  2. Model agirliklarini AlpCAN backend'e entegre et")
print(f"  3. Segmentasyon + malignite pipeline birlestir")
print(f"  4. 3D volumetrik analiz (tam BT serisi)")
print("=" * 70)